***
<a id='beginning'></a>

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [8. 校准](8_0_introduction.ipynb)
    * 建议先阅读： [8.1 作为最小二乘问题的校准](8_1_calibration_least_squares_problem.ipynb)

***


导入标准模块:


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = None
    display = None

STYLE_PATH = Path("../style/course.css")
TOGGLE_PATH = Path("../style/code_toggle.html")

if HTML is not None and display is not None:
    if STYLE_PATH.exists():
        display(HTML(f"<style>{STYLE_PATH.read_text(encoding='utf-8')}</style>"))
    if TOGGLE_PATH.exists():
        display(HTML(TOGGLE_PATH.read_text(encoding="utf-8")))

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=3, suppress=True)

RNG = np.random.default_rng(20260419)


## 校准习题

这一份习题本不再只围绕单一编程题，而是把第 8 章的核心概念拆成一组训练任务。建议按顺序完成：

1. 看懂最小二乘目标函数与参考天线；
2. 观察模型不完整时残差为何无法继续下降；
3. 自己尝试实现一种交替增益求解器，并与 LM 风格求解做比较。

习题的目标不是背命令，而是建立“观测数据、天空模型、求解器和残差”之间的因果关系。


In [ ]:
def baseline_pairs(nant):
    return [(p, q) for p in range(nant) for q in range(p + 1, nant)]


def ew_uv_tracks(ant_x_m, hour_angle_h, dec_deg, wavelength_m=0.214):
    pairs = baseline_pairs(len(ant_x_m))
    hour_angle_rad = np.deg2rad(15.0 * hour_angle_h)
    dec = np.deg2rad(dec_deg)
    u = np.zeros((hour_angle_h.size, len(pairs)))
    v = np.zeros_like(u)

    for ti, ha in enumerate(hour_angle_rad):
        for bi, (p, q) in enumerate(pairs):
            baseline_lambda = (ant_x_m[q] - ant_x_m[p]) / wavelength_m
            u[ti, bi] = baseline_lambda * np.sin(ha)
            v[ti, bi] = -baseline_lambda * np.sin(dec) * np.cos(ha)

    return pairs, u, v


def sky_model_vis(u, fluxes, ls):
    vis = np.zeros_like(u, dtype=complex)
    for flux, ll in zip(fluxes, ls):
        vis += flux * np.exp(-2j * np.pi * u * ll)
    return vis


def gain_track(times_h, nant):
    ant_phase = np.linspace(0.0, np.pi, nant, endpoint=False)
    amp = 1.0 + 0.06 * np.sin(0.9 * times_h[:, None] + 0.6 * ant_phase[None, :])
    phase = 0.45 * np.sin(1.7 * times_h[:, None] + ant_phase[None, :])
    return amp * np.exp(1j * phase)


def apply_gains(model_vis, gains, pairs, noise_std=0.0, rng=None):
    data = np.zeros_like(model_vis, dtype=complex)
    for bi, (p, q) in enumerate(pairs):
        data[:, bi] = gains[:, p] * model_vis[:, bi] * np.conj(gains[:, q])
    if noise_std > 0.0:
        if rng is None:
            rng = np.random.default_rng(0)
        data += noise_std * (
            rng.normal(size=data.shape) + 1j * rng.normal(size=data.shape)
        )
    return data


def pack_params(gains, ref_ant=0):
    log_amp = np.log(np.maximum(np.abs(gains), 1e-12))
    phase = np.angle(gains)
    params = list(log_amp)
    for ant in range(len(gains)):
        if ant == ref_ant:
            continue
        params.append(phase[ant] - phase[ref_ant])
    return np.array(params, dtype=float)


def unpack_params(params, nant, ref_ant=0):
    log_amp = np.array(params[:nant])
    phase = np.zeros(nant)
    idx = nant
    for ant in range(nant):
        if ant == ref_ant:
            continue
        phase[ant] = params[idx]
        idx += 1
    return np.exp(log_amp + 1j * phase)


def predict_vis(model_vec, pairs, gains):
    pred = np.zeros_like(model_vec, dtype=complex)
    for bi, (p, q) in enumerate(pairs):
        pred[bi] = gains[p] * model_vec[bi] * np.conj(gains[q])
    return pred


def residual_vector(params, model_vec, data_vec, pairs, nant, ref_ant=0):
    gains = unpack_params(params, nant, ref_ant=ref_ant)
    resid = data_vec - predict_vis(model_vec, pairs, gains)
    return np.concatenate([resid.real, resid.imag])


def numerical_jacobian(fun, params, eps=1e-6):
    base = fun(params)
    jac = np.zeros((base.size, params.size))
    for idx in range(params.size):
        step = eps * max(1.0, abs(params[idx]))
        shifted = params.copy()
        shifted[idx] += step
        jac[:, idx] = (fun(shifted) - base) / step
    return jac, base


def lm_solve(data_vec, model_vec, pairs, nant, ref_ant=0, n_iter=12, lambda0=1e-2):
    params = pack_params(np.ones(nant, dtype=complex), ref_ant=ref_ant)
    lam = lambda0
    history = []

    def fun(x):
        return residual_vector(x, model_vec, data_vec, pairs, nant, ref_ant=ref_ant)

    for _ in range(n_iter):
        jac, resid = numerical_jacobian(fun, params)
        sse = float(resid @ resid)
        history.append(np.sqrt(np.mean(resid**2)))

        jtj = jac.T @ jac
        grad = jac.T @ resid
        damping = np.diag(np.diag(jtj)) + 1e-9 * np.eye(jtj.shape[0])
        delta = np.linalg.solve(jtj + lam * damping, -grad)

        trial = params + delta
        resid_trial = fun(trial)
        if float(resid_trial @ resid_trial) < sse:
            params = trial
            lam *= 0.7
        else:
            lam *= 2.5

    history.append(np.sqrt(np.mean(fun(params) ** 2)))
    return unpack_params(params, nant, ref_ant=ref_ant), np.array(history)


def solve_track(data, model, pairs, nant, ref_ant=0, n_iter=12):
    gains = np.zeros((data.shape[0], nant), dtype=complex)
    histories = []
    for t in range(data.shape[0]):
        gains[t], hist = lm_solve(
            data[t],
            model[t],
            pairs,
            nant,
            ref_ant=ref_ant,
            n_iter=n_iter,
        )
        histories.append(hist)
    return gains, np.array(histories)


def correct_vis(data, gains, pairs):
    corrected = np.zeros_like(data, dtype=complex)
    for bi, (p, q) in enumerate(pairs):
        corrected[:, bi] = data[:, bi] / (
            gains[:, p] * np.conj(gains[:, q]) + 1e-12
        )
    return corrected


def rms_complex(arr):
    return np.sqrt(np.mean(np.abs(arr) ** 2))


### 练习前的统一数据集

下面的设置与 [8.1](8_1_calibration_least_squares_problem.ipynb) 保持一致。先执行这段代码，后续各题都基于同一组合成数据。


In [ ]:
nant = 5
ant_x_m = np.array([0.0, 36.0, 102.0, 210.0, 348.0])
times_h = np.linspace(-3.5, 3.5, 16)
pairs, u, v = ew_uv_tracks(ant_x_m, times_h, dec_deg=50.0, wavelength_m=0.214)

fluxes_true = np.array([1.0, 0.35, 0.18])
ls_true = np.array([0.0, 0.012, -0.021])
model_vis_true = sky_model_vis(u, fluxes_true, ls_true)

true_gains = gain_track(times_h, nant)
data_vis = apply_gains(model_vis_true, true_gains, pairs, noise_std=0.02, rng=RNG)

solved_ref0, history_ref0 = solve_track(
    data_vis,
    model_vis_true,
    pairs,
    nant,
    ref_ant=0,
    n_iter=12,
)
corrected_ref0 = correct_vis(data_vis, solved_ref0, pairs)


### A. 观察阻尼参数与收敛历史

请修改 `lm_solve()` 中的 `lambda0` 或 `n_iter`，比较下列问题：

- 阻尼过小会不会让第一步更新过于激进？
- 阻尼过大时，收敛会不会变得很慢？
- 对当前数据集，多少步之后收益开始明显变小？

建议你至少尝试三组 `lambda0`，并画出平均收敛历史。


In [ ]:
# 在这里编写你的实验代码。
# 建议：循环多组 lambda0，重复 solve_track，然后比较 history 的均值曲线。


### B. 参考天线与规范选择

请把参考天线从 `0` 改成 `2` 或 `4`，验证下面两句话：

- 求解得到的相位轨迹会改变；
- 但改正后的可见度不应发生实质变化。

你可以量化 `corrected_ref0` 与新结果之间的 RMS 差异。


In [ ]:
# 在这里编写你的代码。
# 提示：调用 solve_track(..., ref_ant=2) 后，再用 correct_vis 比较结果。


### C. 故意删掉一个弱源，观察模型不完整效应

请把第三个弱源从模型中去掉，再比较：

- 收敛历史是否明显停在更高的平台；
- 改正后数据对“真模型”的残差是否明显增加；
- 求解器是否试图用更平滑的增益去吸收模型错误。


In [ ]:
# 在这里编写你的代码。
# 提示：可以复用 8.1 里的思路，构造一个 incomplete model 再重复求解。


### D. 实现一种交替增益更新

下面给出一个交替求解器的骨架。请你补全 `TODO` 部分，实现一种逐天线更新的方向无关增益求解器，并与 `lm_solve` 的结果比较。

建议比较：

- 收敛速度；
- 对初始值的敏感程度；
- 在模型不完整时是否同样会把天空误差吸收入增益。


In [ ]:
def solve_alternating(data, model, pairs, nant, ref_ant=0, n_iter=20):
    gains = np.ones((data.shape[0], nant), dtype=complex)

    # TODO:
    # 1. 按时间切片循环；
    # 2. 对每个天线利用其余天线当前解更新 g_p；
    # 3. 固定参考天线相位；
    # 4. 返回 gains。

    return gains


### E. 写一个校准诊断结论

请用 5 到 8 句话总结你在本习题中看到的现象。建议至少回答：

- 为什么最小二乘目标函数需要参考天线；
- 为什么“收敛了”不代表“模型正确”；
- 在真实数据处理中，哪些残差迹象会让你怀疑 sky model 不完整。


In [ ]:
reference_summary = {
    "raw_to_true_rms": rms_complex(data_vis - model_vis_true),
    "lm_corrected_to_true_rms": rms_complex(corrected_ref0 - model_vis_true),
    "mean_final_history": history_ref0[:, -1].mean(),
}

print("供核对的参考量：")
for key, value in reference_summary.items():
    print(f"  {key}: {value:.4f}")
